# Iris Flower Classification

This notebook develops a supervised machine learning workflow for predicting Iris species from sepal and petal measurements. The current section covers dataset inspection and exploratory data analysis.

## 1. Setup and data loading

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOURCE_DIR = PROJECT_ROOT / "src"
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from iris_classifier.data import FEATURE_COLUMNS, load_iris_dataframe, summarize_dataset
from iris_classifier.eda import correlation_matrix, create_correlation_heatmap, create_pairplot
from iris_classifier.evaluation import (
    create_coefficient_heatmap,
    create_confusion_matrix_plot,
    evaluate_model,
    extract_standardized_coefficients,
    load_model_bundle,
    predict_species,
    save_model_bundle,
)
from iris_classifier.modeling import (
    build_model_candidates,
    compare_models,
    create_model_comparison_plot,
    select_best_model_name,
    split_dataset,
    train_selected_model,
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

iris_df = load_iris_dataframe()
iris_df.head()

## 2. Dataset inspection

In [ ]:
summarize_dataset(iris_df)

In [ ]:
iris_df.loc[:, list(FEATURE_COLUMNS)].describe().round(2)

In [ ]:
iris_df["species"].value_counts().rename_axis("species").to_frame("samples")

The dataset contains 150 complete observations, four numeric features, and three balanced classes with 50 flowers per species.

## 3. Feature relationships

In [ ]:
pairplot_path = create_pairplot(iris_df, OUTPUT_DIR / "pairplot.png")
display(Image(filename=pairplot_path))

Setosa forms a clearly separated group in the petal measurements. Versicolor and Virginica overlap more, although their petal dimensions still provide useful separation. Sepal measurements alone show greater overlap between the species.

## 4. Feature correlations

In [ ]:
correlation_matrix(iris_df).round(2)

In [ ]:
heatmap_path = create_correlation_heatmap(iris_df, OUTPUT_DIR / "correlation_heatmap.png")
display(Image(filename=heatmap_path))

Petal length and petal width have the strongest positive relationship. Sepal length also increases with both petal measurements, while sepal width has weaker relationships with the remaining features.

## 5. Training and test split

In [ ]:
X_train, X_test, y_train, y_test = split_dataset(iris_df)

split_summary = pd.DataFrame({
    "training": y_train.value_counts().sort_index(),
    "testing": y_test.value_counts().sort_index(),
}).rename(index=dict(enumerate(["setosa", "versicolor", "virginica"])))

print(f"Training shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")
split_summary

The 80/20 split is stratified, leaving 40 training samples and 10 test samples for each species. The test set remains untouched during model comparison.

## 6. Model comparison

In [ ]:
model_candidates = build_model_candidates()
comparison_results = compare_models(model_candidates, X_train, y_train)
comparison_results.round(4)

In [ ]:
comparison_plot_path = create_model_comparison_plot(
    comparison_results,
    OUTPUT_DIR / "model_comparison.png",
)
display(Image(filename=comparison_plot_path))

In [ ]:
selected_model_name = select_best_model_name(comparison_results)
selected_model = train_selected_model(
    selected_model_name,
    model_candidates,
    X_train,
    y_train,
)
print(f"Selected model: {selected_model_name}")

All three candidates achieved the same mean five-fold validation accuracy of 95.83% in this run. Logistic Regression is selected through the documented tie-break because it provides a simple linear baseline. Scaling is included inside the Logistic Regression and KNN pipelines so that preprocessing is fitted separately within every training fold.

## 7. Held-out test evaluation

In [ ]:
evaluation = evaluate_model(selected_model, X_test, y_test)
print(f"Held-out test accuracy: {evaluation.accuracy:.2%}")
evaluation.classification_report.round(3)

In [ ]:
confusion_matrix_path = create_confusion_matrix_plot(
    evaluation,
    OUTPUT_DIR / "confusion_matrix.png",
)
display(Image(filename=confusion_matrix_path))

The model correctly classified 28 of 30 test flowers for an accuracy of 93.33%. All 10 Setosa samples were correct. One Versicolor was predicted as Virginica, and one Virginica was predicted as Versicolor.

## 8. Model interpretation

In [ ]:
coefficient_table = extract_standardized_coefficients(selected_model)
coefficient_table.pivot(
    index="species",
    columns="feature",
    values="coefficient",
).round(3)

In [ ]:
coefficient_plot_path = create_coefficient_heatmap(
    coefficient_table,
    OUTPUT_DIR / "coefficient_heatmap.png",
)
display(Image(filename=coefficient_plot_path))

Because the features were standardized before fitting, coefficient magnitudes can be compared within each species. Positive values increase the score for a species and negative values decrease it; they should not be interpreted as causal effects.

## 9. Save the model and make an example prediction

In [ ]:
MODEL_DIR = PROJECT_ROOT / "models"
model_path = save_model_bundle(
    selected_model,
    MODEL_DIR / "iris_classifier.joblib",
)
saved_bundle = load_model_bundle(model_path)
example_measurements = [5.1, 3.5, 1.4, 0.2]
example_prediction = predict_species(saved_bundle, example_measurements)
print(f"Example measurements: {example_measurements}")
print(f"Predicted species: {example_prediction}")

## 10. Conclusion

Logistic Regression provided a compact multiclass classifier after tying with Decision Tree and KNN at 95.83% mean training-set cross-validation accuracy. It correctly classified 28 of 30 previously untouched test samples, producing 93.33% test accuracy. This small educational dataset demonstrates a complete supervised-learning workflow, but the result should not be treated as evidence of performance on broader real-world flower populations.